# Tutorial: MILK sampling

We use [scanpy](https://scanpy.readthedocs.io/en/stable/) to read/write 10X data. Import numpy and scanpy in addlition to screcode.

In [ ]:
import sys
sys.path.append("../..")

In [ ]:
import warnings

import numpy as np
import scanpy as sc
import screcode

warnings.simplefilter('ignore')

Read in the count matrix into an [AnnData](https://anndata.readthedocs.io/en/latest/) object. 

In [ ]:
input_filename = 'data/10k_PBMC_3p_nextgem_Chromium_Controller_filtered_feature_bc_matrix.h5'
adata = sc.read_10x_h5(input_filename)
adata.layers["Raw"] = adata.X.toarray()
adata

## Preprocess the data with Scanpy

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata, chunked=False, zero_center=False)
sc.pp.neighbors(adata, n_pcs=50)
sc.tl.leiden(adata, resolution=0.3)
sc.tl.umap(adata)

## Sampling

Sample 2000 data points from `adata`. A random sample is also taken for comparison.

In [ ]:
recode = screcode.RECODE()

In [ ]:
n_samples = 2000
milk_idx = recode.milk_sampling(adata, n_samples=n_samples, thresh_percentile=1)
rand_idx = np.sort(np.random.choice(adata.X.shape[0], n_samples, replace=False))

In [ ]:
print(milk_idx)
print(rand_idx)

## Result

In [ ]:
milk_adata = adata[milk_idx]
rand_adata = adata[rand_idx]

### PCA

In [ ]:
sc.pl.pca(adata, components=['1,2', '3,4', '5,6'], color=['leiden'], size=10)
sc.pl.pca(milk_adata, components=['1,2','3,4','5,6'], color=["leiden"], size=10)
sc.pl.pca(rand_adata, components=['1,2','3,4','5,6'], color=["leiden"], size=10)

### UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden'],legend_loc='on data', size=30)
sc.pl.umap(milk_adata, color=['leiden'],legend_loc='on data', size=30)
sc.pl.umap(rand_adata, color=['leiden'],legend_loc='on data', size=30)

### Cell count

In [ ]:
import pandas as pd
counts = [ad.obs["leiden"].value_counts().rename_axis(None) for ad in [adata, milk_adata, rand_adata]]
df = pd.concat(counts, axis=1, keys=["Total", "Milk sampling", "Random sampling"])
print("Cell count in each cluster:")
df